In [29]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

In [30]:
train_df = pd.read_csv("../data/train_processed.csv")
test_df = pd.read_csv("../data/test_processed.csv")

In [31]:
features = [
    'T2M', 'RH2M', 'PS',
    'hour', 'month',
    'T2M_diff', 'RH2M_diff', 'PS_diff',
    'T2M_roll_mean', 'RH2M_roll_mean', 'PS_roll_mean',
    'T2M_roll_std', 'RH2M_roll_std', 'PS_roll_std',
    'T2M_roll_dev', 'RH2M_roll_dev', 'PS_roll_dev'
]

X_train = train_df[features]

X_test = test_df[features]
y_test = test_df['is_anomaly']

In [32]:
model = IsolationForest(
    contamination=0.03,
    random_state=42
)

model.fit(X_train)

pred = model.predict(X_test)
y_pred = (pred == -1).astype(int)

results = test_df.copy()
results['predicted_anomaly'] = y_pred

for anomaly in results['anomaly_type'].unique():
    subset = results[results['anomaly_type'] == anomaly]

    if anomaly != 'normal':
        detected = subset['predicted_anomaly'].mean()

        print(
            anomaly,
            "->",
            round(detected * 100, 2),
            "% detected"
        )

temperature_spike -> 90.16 % detected
multivariate_inconsistency -> 85.33 % detected
temperature_frozen -> 17.71 % detected
temperature_drift -> 16.13 % detected


In [33]:
test_df['T2M_frozen_4'] = (
    test_df['T2M'].rolling(4).std() < 0.05
)

In [47]:
frozen_rows = test_df[
    test_df['anomaly_type'] == 'temperature_frozen'
]

print(
    frozen_rows['T2M_frozen_4'].mean() * 100,
    "% frozen anomalies detected"
)

71.875 % frozen anomalies detected


In [39]:
full_test = pd.read_csv(
    "../data/test_anomalies_full.csv",
    parse_dates=["datetime"]
)

In [40]:
full_test['communication_error'] = (
    full_test[['T2M', 'RH2M', 'PS']]
    .isna()
    .any(axis=1)
)

In [42]:
comm_rows = full_test[
    full_test['anomaly_type'] == 'communication_error'
]

print(
    comm_rows['communication_error'].mean() * 100,
    "% communication errors detected"
)

100.0 % communication errors detected


In [43]:
def detect_status(row, recent_temps, model, features):
    # 1. Communication error
    if row[['T2M', 'RH2M', 'PS']].isna().any():
        return {
            'status': 'anomaly',
            'type': 'communication_error',
            'severity': 'high',
            'reason': 'Missing sensor reading'
        }

    # 2. Frozen sensor
    if len(recent_temps) >= 6 and recent_temps[-6:].std() < 0.05:
        return {
            'status': 'anomaly',
            'type': 'temperature_frozen',
            'severity': 'medium',
            'reason': 'Temperature barely changed for 6 readings'
        }

    # 3. Isolation Forest
    x = row[features].values.reshape(1, -1)
    pred = model.predict(x)[0]

    if pred == -1:
        return {
            'status': 'anomaly',
            'type': 'ml_anomaly',
            'severity': 'medium',
            'reason': 'Unusual weather-sensor pattern detected'
        }


In [48]:
normal_rows = test_df[
    test_df['anomaly_type'] == 'normal'
]

print(
    normal_rows['T2M_frozen_4'].mean() * 100,
    "% normal rows falsely flagged"
)

0.025879917184265012 % normal rows falsely flagged


In [49]:
import joblib

joblib.dump(model, "../models/isolation_forest.pkl")

['../models/isolation_forest.pkl']